In [1]:
"""
Bird Classification using CUB-200-2011 Dataset
Models: ResNet50 & EfficientNet-B0
Modes:  Transfer Learning  (frozen backbone)
        Fine-Tuning        (full unfrozen training)
"""

import os
import random
import shutil
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
# ─────────────────────────── CONFIG ────────────────────────────
DATASET_ROOT  = "./CUB_200_2011"          # path to extracted dataset
OUTPUT_DIR    = "./outputs"
NUM_CLASSES   = 6                          # select first 6 species
BATCH_SIZE    = 32
IMAGE_SIZE    = 224
SEED          = 42

# Training epochs per phase
TL_EPOCHS     = 10    # Transfer Learning  (frozen)
FT_EPOCHS     = 15    # Fine-Tuning        (unfrozen, lower LR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Using device: cuda


In [3]:
# ─────────────────────────── DATASET ───────────────────────────
class CUB200Dataset(Dataset):
    """
    Loads a subset of CUB-200-2011.

    Directory structure expected:
        DATASET_ROOT/
            images/
                001.Black_footed_Albatross/  *.jpg
                002.Laysan_Albatross/        *.jpg
                ...
            images.txt
            image_class_labels.txt
            train_test_split.txt
            classes.txt
    """

    def __init__(self, root, split="train", num_classes=6, transform=None):
        self.root      = root
        self.transform = transform

        # ── parse metadata ──────────────────────────────────────
        images_df = pd.read_csv(
            os.path.join(root, "images.txt"),
            sep=" ", names=["image_id", "image_path"])

        labels_df = pd.read_csv(
            os.path.join(root, "image_class_labels.txt"),
            sep=" ", names=["image_id", "class_id"])

        split_df = pd.read_csv(
            os.path.join(root, "train_test_split.txt"),
            sep=" ", names=["image_id", "is_train"])

        classes_df = pd.read_csv(
            os.path.join(root, "classes.txt"),
            sep=" ", names=["class_id", "class_name"])

        # merge into one frame
        df = images_df.merge(labels_df, on="image_id") \
                      .merge(split_df,  on="image_id")

        # keep only first num_classes species (class_id 1 … num_classes)
        df = df[df["class_id"] <= num_classes].copy()

        # remap class_id → 0-indexed label
        self.class_names = (
            classes_df[classes_df["class_id"] <= num_classes]["class_name"]
            .apply(lambda x: x.split(".")[-1].replace("_", " "))
            .tolist()
        )
        df["label"] = df["class_id"] - 1

        # train / test split
        is_train = 1 if split == "train" else 0
        df = df[df["is_train"] == is_train].reset_index(drop=True)

        self.paths  = df["image_path"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.paths[idx])
        image    = Image.open(img_path).convert("RGB")
        label    = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [4]:
# ─────────────────────────── TRANSFORMS ────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [17]:
def get_loaders(root, num_classes=NUM_CLASSES, batch_size=BATCH_SIZE):
    train_ds = CUB200Dataset(root, split="train",
                             num_classes=num_classes,
                             transform=train_transform)
    val_ds = CUB200Dataset(root, split="test",
                           num_classes=num_classes,
                           transform=val_transform)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=torch.cuda.is_available()
    )

    print(f"\nDataset  ▸  train={len(train_ds)}  val={len(val_ds)}")
    print(f"Classes  ▸  {train_ds.class_names}\n")
    return train_loader, val_loader, train_ds.class_names

In [6]:
# ─────────────────────────── MODELS ────────────────────────────
def get_model_resnet(num_classes=NUM_CLASSES, mode="transfer"):
    """ResNet-50. mode ∈ {'transfer', 'finetune'}"""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    if mode == "transfer":
        for param in model.parameters():
            param.requires_grad = False           # freeze everything

    # replace classifier head
    model.fc = nn.Sequential(
        nn.Linear(2048, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes),
    )
    return model.to(DEVICE)


def get_model_efficientnet(num_classes=NUM_CLASSES, mode="transfer"):
    """EfficientNet-B0. mode ∈ {'transfer', 'finetune'}"""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if mode == "transfer":
        for param in model.parameters():
            param.requires_grad = False

    # replace classifier head
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model.to(DEVICE)

In [7]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, correct / total

In [8]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)
        all_preds .extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels

In [9]:
def train_model(model, train_loader, val_loader,
                epochs, lr, label, scheduler_type="cosine"):
    """
    Full training loop with scheduler.
    Returns history dict and (y_true, y_pred) from last epoch.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )

    if scheduler_type == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    else:
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

    print(f"\n{'='*55}")
    print(f"  {label}  |  epochs={epochs}  lr={lr}")
    print(f"{'='*55}")
    print(f"{'Ep':>4}  {'T-Loss':>8}  {'T-Acc':>7}  {'V-Loss':>8}  {'V-Acc':>7}  {'LR':>9}")
    print(f"{'-'*55}")

    best_val_acc = 0
    best_state   = None
    y_true = y_pred = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta             = train_one_epoch(model, train_loader, criterion, optimizer)
        vl, va, y_pred, y_true = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tl)
        history["train_acc"] .append(ta)
        history["val_loss"]  .append(vl)
        history["val_acc"]   .append(va)

        cur_lr = optimizer.param_groups[0]["lr"]
        mark   = " ✓" if va > best_val_acc else ""
        print(f"{epoch:>4}  {tl:>8.4f}  {ta*100:>6.2f}%  "
              f"{vl:>8.4f}  {va*100:>6.2f}%  {cur_lr:>9.2e}{mark}")

        if va > best_val_acc:
            best_val_acc = va
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"\n  Best val acc: {best_val_acc*100:.2f}%")
    return history, y_true, y_pred

In [10]:
# ─────────────────────────── PLOTTING ──────────────────────────
def plot_history(histories, title, save_path):
    """Plot loss & accuracy curves for multiple runs."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    for i, (label, h) in enumerate(histories.items()):
        c  = colors[i % len(colors)]
        ep = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(ep, h["train_loss"], label=f"{label} Train", color=c, linestyle="--")
        axes[0].plot(ep, h["val_loss"],   label=f"{label} Val",   color=c)
        axes[1].plot(ep, [a*100 for a in h["train_acc"]], linestyle="--", color=c, label=f"{label} Train")
        axes[1].plot(ep, [a*100 for a in h["val_acc"]],                   color=c, label=f"{label} Val")

    axes[0].set(title="Loss",     xlabel="Epoch", ylabel="CE Loss")
    axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {save_path}")


def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {save_path}")


def plot_comparison_bar(results, save_path):
    """Bar chart comparing final val-acc across all experiments."""
    labels = list(results.keys())
    accs   = [v * 100 for v in results.values()]
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(labels, accs, color=colors[:len(labels)], edgecolor="white", width=0.5)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{acc:.2f}%", ha="center", va="bottom", fontweight="bold")

    ax.set(ylabel="Validation Accuracy (%)", title="Model Comparison – Final Val Accuracy",
           ylim=(0, 105))
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {save_path}")

In [16]:
# ─────────────────────────── MAIN ──────────────────────────────
def main():
    # ── 1. Data ─────────────────────────────────────────────────
    train_loader, val_loader, class_names = get_loaders(DATASET_ROOT)

    all_histories = {}
    final_results = {}

    # ── 2. ResNet50 – Transfer Learning ─────────────────────────
    print("\n>>> ResNet-50  |  Transfer Learning (frozen backbone)")
    model_rn_tl = get_model_resnet(mode="transfer")
    h_rn_tl, yt_rn_tl, yp_rn_tl = train_model(
        model_rn_tl, train_loader, val_loader,
        epochs=TL_EPOCHS, lr=1e-3, label="ResNet50 TL")

    all_histories["ResNet50-TL"]   = h_rn_tl
    final_results["ResNet50-TL"]   = max(h_rn_tl["val_acc"])

    plot_confusion_matrix(yt_rn_tl, yp_rn_tl, class_names,
                          "ResNet50 – Transfer Learning",
                          f"{OUTPUT_DIR}/cm_resnet50_tl.png")
    print(classification_report(yt_rn_tl, yp_rn_tl, target_names=class_names))

    # ── 3. ResNet50 – Fine-Tuning ────────────────────────────────
    print("\n>>> ResNet-50  |  Fine-Tuning (all layers unfrozen)")
    # Start from the TL weights → then unfreeze
    for param in model_rn_tl.parameters():
        param.requires_grad = True

    h_rn_ft, yt_rn_ft, yp_rn_ft = train_model(
        model_rn_tl, train_loader, val_loader,
        epochs=FT_EPOCHS, lr=1e-4, label="ResNet50 FT")  # lower LR!

    all_histories["ResNet50-FT"]   = h_rn_ft
    final_results["ResNet50-FT"]   = max(h_rn_ft["val_acc"])

    plot_confusion_matrix(yt_rn_ft, yp_rn_ft, class_names,
                          "ResNet50 – Fine-Tuning",
                          f"{OUTPUT_DIR}/cm_resnet50_ft.png")
    print(classification_report(yt_rn_ft, yp_rn_ft, target_names=class_names))

    # ── 4. EfficientNet-B0 – Transfer Learning ───────────────────
    print("\n>>> EfficientNet-B0  |  Transfer Learning (frozen backbone)")
    model_en_tl = get_model_efficientnet(mode="transfer")
    h_en_tl, yt_en_tl, yp_en_tl = train_model(
        model_en_tl, train_loader, val_loader,
        epochs=TL_EPOCHS, lr=1e-3, label="EffNet-B0 TL")

    all_histories["EffNet-TL"]     = h_en_tl
    final_results["EffNet-TL"]     = max(h_en_tl["val_acc"])

    plot_confusion_matrix(yt_en_tl, yp_en_tl, class_names,
                          "EfficientNet-B0 – Transfer Learning",
                          f"{OUTPUT_DIR}/cm_effnet_tl.png")
    print(classification_report(yt_en_tl, yp_en_tl, target_names=class_names))

    # ── 5. EfficientNet-B0 – Fine-Tuning ────────────────────────
    print("\n>>> EfficientNet-B0  |  Fine-Tuning (all layers unfrozen)")
    for param in model_en_tl.parameters():
        param.requires_grad = True

    h_en_ft, yt_en_ft, yp_en_ft = train_model(
        model_en_tl, train_loader, val_loader,
        epochs=FT_EPOCHS, lr=5e-5, label="EffNet-B0 FT")

    all_histories["EffNet-FT"]     = h_en_ft
    final_results["EffNet-FT"]     = max(h_en_ft["val_acc"])

    plot_confusion_matrix(yt_en_ft, yp_en_ft, class_names,
                          "EfficientNet-B0 – Fine-Tuning",
                          f"{OUTPUT_DIR}/cm_effnet_ft.png")
    print(classification_report(yt_en_ft, yp_en_ft, target_names=class_names))

    # ── 6. Plots ─────────────────────────────────────────────────
    plot_history(
        {"ResNet50-TL": h_rn_tl, "ResNet50-FT": h_rn_ft},
        "ResNet-50: Transfer Learning vs Fine-Tuning",
        f"{OUTPUT_DIR}/curves_resnet50.png"
    )
    plot_history(
        {"EffNet-TL": h_en_tl, "EffNet-FT": h_en_ft},
        "EfficientNet-B0: Transfer Learning vs Fine-Tuning",
        f"{OUTPUT_DIR}/curves_effnet.png"
    )
    plot_comparison_bar(final_results, f"{OUTPUT_DIR}/comparison_bar.png")

    # ── 7. Summary ───────────────────────────────────────────────
    print("\n" + "="*55)
    print("  FINAL RESULTS SUMMARY")
    print("="*55)
    print(f"{'Experiment':<20}  {'Best Val Acc':>12}")
    print("-"*35)
    for k, v in final_results.items():
        print(f"{k:<20}  {v*100:>11.2f}%")
    best = max(final_results, key=final_results.get)
    print(f"\n  🏆 Winner: {best}  ({final_results[best]*100:.2f}%)")
    print("="*55)

    # ── 8. Save best model ───────────────────────────────────────
    models_dict = {
        "ResNet50-TL": model_rn_tl,
        "ResNet50-FT": model_rn_tl,
        "EffNet-TL": model_en_tl,
        "EffNet-FT": model_en_tl,
    }
    best_model = models_dict[best]
    torch.save(best_model.state_dict(), f"{OUTPUT_DIR}/best_model.pth")

In [18]:
if __name__ == "__main__":
    main()


Dataset  ▸  train=180  val=143
Classes  ▸  ['Black footed Albatross', 'Laysan Albatross', 'Sooty Albatross', 'Groove billed Ani', 'Crested Auklet', 'Least Auklet']


>>> ResNet-50  |  Transfer Learning (frozen backbone)

  ResNet50 TL  |  epochs=10  lr=0.001
  Ep    T-Loss    T-Acc    V-Loss    V-Acc         LR
-------------------------------------------------------
   1    1.8372   17.78%    1.3801   51.75%   9.76e-04 ✓
   2    1.3351   54.44%    1.0829   68.53%   9.05e-04 ✓
   3    1.0610   61.11%    0.8222   72.03%   7.94e-04 ✓
   4    1.0209   62.22%    0.7434   80.42%   6.55e-04 ✓
   5    0.7823   76.67%    0.7077   79.02%   5.00e-04
   6    0.7346   78.89%    0.6130   81.12%   3.45e-04 ✓
   7    0.6624   81.11%    0.5842   84.62%   2.06e-04 ✓
   8    0.5883   86.67%    0.5693   83.22%   9.55e-05
   9    0.5686   84.44%    0.5643   83.22%   2.45e-05
  10    0.5828   82.78%    0.5630   83.22%   0.00e+00

  Best val acc: 84.62%
  Saved → ./outputs/cm_resnet50_tl.png
               